## Program 2B

CCD theory applied with median radius spatial hashing using an elevated capsule (swept sphere). Imports data from Program 2A's CSV files.

In [22]:
# Import Libraries

import pandas as pd
import numpy as np
import time
from time import perf_counter_ns
from collections import defaultdict
from pathlib import Path
import re
import math
import gc

In [23]:
# Import A and A_ref

folder_path = r"C:\Users\Smith\OneDrive\MSc Project\05 Final Code"

A_COLS = {
    "x": 0, "y": 1, "z": 2,
    "x+": 3, "y+": 4, "z+": 5,
    "i3": 6, "j3": 7, "k3": 8,
    "h3": 9, "h4": 10, "r3": 11, "r4": 12,
    "q1": 13, "q2": 14, "q3": 15,
    "delta_seg": 16,
}

A_REF_COLS = {
    "x": 0, "y": 1, "z": 2,
    "i": 3, "j": 4, "k": 5,
    "layer_end": 6,
}

def load_csv_as_array(path):
    skiprows = 0
    arr = np.loadtxt(path, delimiter=",", skiprows=skiprows)

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    return arr

def load_inputs(folder_path):
    folder = Path(folder_path)

    A_path = folder / "A.csv"
    A_ref_path = folder / "A_ref.csv"

    if not A_path.exists():
        raise SystemExit(f"Could not find: {A_path}")

    if not A_ref_path.exists():
        raise SystemExit(f"Could not find: {A_ref_path}")

    A = load_csv_as_array(A_path)
    A_ref = load_csv_as_array(A_ref_path)

    required_A_cols = max(A_COLS.values()) + 1
    required_A_ref_cols = max(A_REF_COLS.values()) + 1

    return A, A_ref

In [24]:
# User Variables

h_0 = 0.2
EPS = 1e-12
TOL = 1e-12

In [25]:
# PIT Helping Functions

def q1_to_ref_index(q1_value):

    if not np.isfinite(q1_value):
        return None

    idx = int(round(q1_value))
    return idx


def t_interval_for_axial_range(a0, du, s_min, s_max, eps=EPS):

    a0 = np.asarray(a0, dtype=float)
    n = a0.shape[0]

    t_lo = np.zeros(n, dtype=float)
    t_hi = np.ones(n, dtype=float)

    if s_max < s_min:
        return t_lo, t_hi, np.zeros(n, dtype=bool)

    if abs(du) <= eps:
        valid = (a0 >= s_min - eps) & (a0 <= s_max + eps)
        return t_lo, t_hi, valid

    t_a = (a0 - s_max) / du
    t_b = (a0 - s_min) / du

    lo = np.minimum(t_a, t_b)
    hi = np.maximum(t_a, t_b)

    t_lo = np.maximum(0.0, lo)
    t_hi = np.minimum(1.0, hi)

    valid = t_lo <= t_hi + eps

    return t_lo, t_hi, valid


def quadratic_value(Aq, Bq, Cq, t):

    return Aq * t * t + Bq * t + Cq


def quadratic_min_on_interval(Aq, Bq, Cq, t_lo, t_hi, valid, eps=EPS):

    best_t = t_lo.copy()
    best_q = quadratic_value(Aq, Bq, Cq, best_t)

    q_hi = quadratic_value(Aq, Bq, Cq, t_hi)
    use_hi = q_hi < best_q

    best_q[use_hi] = q_hi[use_hi]
    best_t[use_hi] = t_hi[use_hi]

    if Aq > eps:
        t_vertex = -Bq / (2.0 * Aq)

        use_vertex = (
            valid
            & (t_vertex >= t_lo - eps)
            & (t_vertex <= t_hi + eps)
        )

        if np.any(use_vertex):
            t_vertex_clipped = np.clip(t_vertex, t_lo, t_hi)
            q_vertex = quadratic_value(Aq, Bq, Cq, t_vertex_clipped)

            improve = use_vertex & (q_vertex < best_q)

            best_q[improve] = q_vertex[improve]
            best_t[improve] = t_vertex_clipped[improve]

    best_q[~valid] = np.inf
    best_t[~valid] = np.nan

    return best_q, best_t


def ctc_tip_touching_sphere_params(h3, h4, r3, r4, eps=EPS): # Spatial Hashing Broad Phase - Capsule Helpers

    h3 = float(h3)
    h4 = float(h4)
    r3 = float(r3)
    r4 = float(r4)

    H = h3 + h4

    terms = []
    terms.append((r3*r3 + h3*h3) / (2.0*h3))
    
    if h4 > eps:
        terms.append((r4*r4 + h3*h3) / (2.0*h3))
        terms.append((r4*r4 + H*H) / (2.0*H))

    k = max(terms)
    R = k

    return k, R


def choose_median_capsule_hash_cell_size(A, eps=EPS):

    radii = []

    for A_row_idx, row in enumerate(A):
        h3 = row[A_COLS["h3"]]
        h4 = row[A_COLS["h4"]]
        r3 = row[A_COLS["r3"]]
        r4 = row[A_COLS["r4"]]

        try:
            _, R = ctc_tip_touching_sphere_params(
                h3=h3,
                h4=h4,
                r3=r3,
                r4=r4,
                eps=eps,
            )

            if np.isfinite(R) and R > eps:
                radii.append(R)

        except SystemExit:
            pass

    if len(radii) == 0:
        raise SystemExit(
            "Could not choose cell size because no valid capsule radii were found."
        )

    radii = np.asarray(radii, dtype=np.float64)

    cell_size = float(np.median(radii))

    if not np.isfinite(cell_size) or cell_size <= eps:
        raise SystemExit("Chosen median cell size is invalid.")

    print(f"Smallest capsule/sphere radius found: {np.min(radii):.6g}")
    print(f"Median capsule/sphere radius used:    {cell_size:.6g}")
    print(f"Largest capsule/sphere radius found:  {np.max(radii):.6g}")

    return cell_size


def point_cell_key(point_xyz, inv_cell_size): # Converts a point into a spatial hash cell key.

    return tuple(np.floor(point_xyz * inv_cell_size).astype(np.int64)) 


def add_point_to_spatial_hash(grid, point_xyz, point_local_idx, inv_cell_size): # Adds one accumulated A_ref point to the spatial hash

    key = point_cell_key(point_xyz, inv_cell_size)
    grid[key].append(int(point_local_idx))


def point_to_segment_distance_sq(points, a, b, eps=EPS):

    points = np.asarray(points, dtype=np.float64)
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    ab = b - a
    ab_sq = float(np.dot(ab, ab))

    if ab_sq <= eps:
        w = points - a
        return np.einsum("ij,ij->i", w, w)

    ap = points - a
    lam = (ap @ ab) / ab_sq
    lam = np.clip(lam, 0.0, 1.0)

    closest = a + lam[:, None] * ab
    diff = points - closest

    return np.einsum("ij,ij->i", diff, diff)


def capsule_hash_candidate_indices(points, grid, p0, p1, u, h3, h4, r3, r4, cell_size, eps=EPS):

    points = np.asarray(points, dtype=np.float64)

    if points.shape[0] == 0:
        return np.empty(0, dtype=np.int64), 0, 0, np.nan, np.nan

    inv_cell_size = 1.0 / cell_size

    p0 = np.asarray(p0, dtype=np.float64)
    p1 = np.asarray(p1, dtype=np.float64)
    u = np.asarray(u, dtype=np.float64)

    norm_u = np.linalg.norm(u)

    u = u / norm_u

    k_sphere, R_capsule = ctc_tip_touching_sphere_params(
        h3=h3,
        h4=h4,
        r3=r3,
        r4=r4,
        eps=eps,
    )

    # Swept bounding sphere centreline
    c0 = p0 + k_sphere * u
    c1 = p1 + k_sphere * u

    R_query = R_capsule + eps

    # AABB of the capsule
    bb_min = np.minimum(c0, c1) - R_query
    bb_max = np.maximum(c0, c1) + R_query

    cmin = np.floor(bb_min * inv_cell_size).astype(np.int64)
    cmax = np.floor(bb_max * inv_cell_size).astype(np.int64)

    candidate_lists = []

    for cx in range(cmin[0], cmax[0] + 1):
        for cy in range(cmin[1], cmax[1] + 1):
            for cz in range(cmin[2], cmax[2] + 1):

                ids = grid.get((cx, cy, cz))

                if ids:
                    candidate_lists.append(ids)

    if not candidate_lists:
        return np.empty(0, dtype=np.int64), 0, 0, k_sphere, R_capsule

    total_ids = sum(len(ids) for ids in candidate_lists)

    J = np.fromiter(
        (idx for ids in candidate_lists for idx in ids),
        dtype=np.int64,
        count=total_ids,
    )

    broad_cell_candidate_count = J.size

    if J.size == 0:
        return np.empty(0, dtype=np.int64), broad_cell_candidate_count, 0, k_sphere, R_capsule

    # Exact point-vs-capsule broad phase filter.
    Pj = points[J]

    dist_sq = point_to_segment_distance_sq(Pj, c0, c1, eps=eps)

    capsule_ok = dist_sq <= R_query * R_query

    J = J[capsule_ok]

    if J.size > 1:
        J = np.unique(J)

    capsule_candidate_count = J.size

    return J, broad_cell_candidate_count, capsule_candidate_count, k_sphere, R_capsule

In [26]:
# PIT Function

def swept_ctc_hits_points(points, p0, p1, u, h3, h4, r3, r4, h_0=0.0, tol=TOL, candidate_indices=None):

    points = np.asarray(points, dtype=float)
    n = points.shape[0]

    hit = np.zeros(n, dtype=bool)
    best_t = np.full(n, np.nan)
    best_axial = np.full(n, np.nan)
    best_radial = np.full(n, np.nan)
    best_allowed = np.full(n, np.nan)
    best_margin = np.full(n, -np.inf)
    best_section = np.full(n, "", dtype=object)

    empty_details = {
        "t": best_t,
        "axial": best_axial,
        "radial": best_radial,
        "allowed_radius": best_allowed,
        "margin": best_margin,
        "section": best_section,
    }

    if n == 0:
        return hit, empty_details

    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    u = np.asarray(u, dtype=float)

    norm_u = np.linalg.norm(u)

    u = u / norm_u

    h3 = float(h3)
    h4 = float(h4)
    r3 = float(r3)
    r4 = float(r4)
    h_0 = max(0.0, float(h_0))

    H = h3 + h4

    if H <= h_0 + EPS:
        return hit, empty_details

    # Candidate selection from broad phase
    if candidate_indices is None:
        idx = np.arange(n, dtype=np.int64)
        
    else:
        
        idx = np.asarray(candidate_indices, dtype=np.int64)
        idx = idx[(idx >= 0) & (idx < n)]
    
        if idx.size > 1:
            idx = np.unique(idx)
    
    if idx.size == 0:
        return hit, empty_details
    
    pts = points[idx]

    d = p1 - p0

    du = float(np.dot(d, u))
    d_perp = d - du * u
    d_perp_sq = float(np.dot(d_perp, d_perp))

    w = pts - p0

    a0 = w @ u
    w_perp = w - a0[:, None] * u

    w_perp_sq = np.einsum("ij,ij->i", w_perp, w_perp)
    w_perp_dot_d_perp = w_perp @ d_perp

    local_hit = np.zeros(len(idx), dtype=bool)
    local_t = np.full(len(idx), np.nan)
    local_axial = np.full(len(idx), np.nan)
    local_radial = np.full(len(idx), np.nan)
    local_allowed = np.full(len(idx), np.nan)
    local_margin = np.full(len(idx), -np.inf)
    local_section = np.full(len(idx), "", dtype=object)

    # CONE
    cone_s_min = h_0
    cone_s_max = min(h3, H)

    if h3 > EPS and r3 >= 0.0 and cone_s_max >= cone_s_min - EPS:
        k = r3 / h3

        t_lo, t_hi, valid = t_interval_for_axial_range(
            a0=a0,
            du=du,
            s_min=cone_s_min,
            s_max=cone_s_max,
        )

        Aq = d_perp_sq - (k * du) ** 2
        Bq = -2.0 * w_perp_dot_d_perp + 2.0 * (k ** 2) * a0 * du
        Cq = w_perp_sq - (k * a0) ** 2

        q_min, t_min = quadratic_min_on_interval(
            Aq=Aq,
            Bq=Bq,
            Cq=Cq,
            t_lo=t_lo,
            t_hi=t_hi,
            valid=valid,
        )

        cone_hit = q_min <= tol

        if np.any(cone_hit):
            axial = a0 - t_min * du

            radial_sq = (
                w_perp_sq
                - 2.0 * t_min * w_perp_dot_d_perp
                + (t_min ** 2) * d_perp_sq
            )

            radial = np.sqrt(np.maximum(0.0, radial_sq))
            allowed = k * axial
            margin = allowed - radial

            improve = cone_hit & (margin > local_margin)

            local_hit[improve] = True
            local_t[improve] = t_min[improve]
            local_axial[improve] = axial[improve]
            local_radial[improve] = radial[improve]
            local_allowed[improve] = allowed[improve]
            local_margin[improve] = margin[improve]
            local_section[improve] = "cone"

    # CYLINDER
    cyl_s_min = max(h_0, h3)
    cyl_s_max = H

    if h4 > EPS and r4 >= 0.0 and cyl_s_max >= cyl_s_min - EPS:
        t_lo, t_hi, valid = t_interval_for_axial_range(
            a0=a0,
            du=du,
            s_min=cyl_s_min,
            s_max=cyl_s_max,
        )

        Aq = d_perp_sq
        Bq = -2.0 * w_perp_dot_d_perp
        Cq = w_perp_sq

        q_min, t_min = quadratic_min_on_interval(
            Aq=Aq,
            Bq=Bq,
            Cq=Cq,
            t_lo=t_lo,
            t_hi=t_hi,
            valid=valid,
        )

        cyl_hit = q_min <= (r4 * r4 + tol)

        if np.any(cyl_hit):
            axial = a0 - t_min * du
            radial = np.sqrt(np.maximum(0.0, q_min))
            allowed = np.full_like(radial, r4)
            margin = allowed - radial

            improve = cyl_hit & (margin > local_margin)

            local_hit[improve] = True
            local_t[improve] = t_min[improve]
            local_axial[improve] = axial[improve]
            local_radial[improve] = radial[improve]
            local_allowed[improve] = allowed[improve]
            local_margin[improve] = margin[improve]
            local_section[improve] = "cyl"

    hit[idx] = local_hit
    best_t[idx] = local_t
    best_axial[idx] = local_axial
    best_radial[idx] = local_radial
    best_allowed[idx] = local_allowed
    best_margin[idx] = local_margin
    best_section[idx] = local_section

    details = {
        "t": best_t,
        "axial": best_axial,
        "radial": best_radial,
        "allowed_radius": best_allowed,
        "margin": best_margin,
        "section": best_section,
    }

    return hit, details

In [27]:
# Called Function - PIT Looper

def ccd_ctc_with_hash(folder_path, h_0):
    
    A, A_ref = load_inputs(folder_path)

    t_start = perf_counter_ns()
    t_last = t_start

    def lap(name):
        nonlocal t_last
        now = perf_counter_ns()
        dt_ms = (now - t_last) / 1_000_000
        total_ms = (now - t_start) / 1_000_000
        print(f"{name:<45} /{dt_ms:10.3f}/ ms   total: {total_ms:10.3f} ms")
        t_last = now

    lap("A and A_ref Loaded from CSV")

    cell_size = choose_median_capsule_hash_cell_size(A, eps=EPS)

    inv_cell_size = 1.0 / cell_size

    print(f"Capsule spatial hash cell size: {cell_size:.6g}")

    # Maximum possible number of unique A_ref points is A_ref.shape[0]!
    test_point_xyz = np.empty((A_ref.shape[0], 3), dtype=np.float64)
    test_point_ref_rows = np.empty(A_ref.shape[0], dtype=np.int64)

    ref_row_added = np.zeros(A_ref.shape[0], dtype=bool)
    n_test_points = 0

    # Dynamic spatial hash of accumulated A_ref points
    # Each point is inserted once - no duplicates
    grid = defaultdict(list)

    hit_records = []
    skipped_bad_q1 = 0

    broad_cell_candidate_count = 0
    capsule_candidate_count = 0
    exact_test_count = 0

    max_capsule_candidates_one_row = 0
    max_broad_cell_candidates_one_row = 0

    loop_start = perf_counter_ns()

    for A_row_idx in range(A.shape[0]):
        row = A[A_row_idx]

        q1 = row[A_COLS["q1"]]
        q2 = row[A_COLS["q2"]]
        q3 = row[A_COLS["q3"]]

        ref_idx = q1_to_ref_index(q1)

        if ref_idx is None or ref_idx < 0 or ref_idx >= A_ref.shape[0]:
            skipped_bad_q1 += 1
            continue

        if not ref_row_added[ref_idx]:
            ref_row_added[ref_idx] = True

            new_point_xyz = A_ref[
                ref_idx,
                [A_REF_COLS["x"], A_REF_COLS["y"], A_REF_COLS["z"]]
            ].astype(np.float64)

            test_point_ref_rows[n_test_points] = ref_idx
            test_point_xyz[n_test_points, :] = new_point_xyz

            add_point_to_spatial_hash(
                grid=grid,
                point_xyz=new_point_xyz,
                point_local_idx=n_test_points,
                inv_cell_size=inv_cell_size,
            )

            n_test_points += 1

        points_now = test_point_xyz[:n_test_points]
        point_ref_rows_now = test_point_ref_rows[:n_test_points]

        p0 = row[[A_COLS["x"], A_COLS["y"], A_COLS["z"]]]
        p1 = row[[A_COLS["x+"], A_COLS["y+"], A_COLS["z+"]]]

        u = row[[A_COLS["i3"], A_COLS["j3"], A_COLS["k3"]]]

        h3 = row[A_COLS["h3"]]
        h4 = row[A_COLS["h4"]]
        r3 = row[A_COLS["r3"]]
        r4 = row[A_COLS["r4"]]

        # Capsule broad phase

        candidate_indices, broad_count, capsule_count, k_sphere, R_capsule = capsule_hash_candidate_indices(
            points=points_now,
            grid=grid,
            p0=p0,
            p1=p1,
            u=u,
            h3=h3,
            h4=h4,
            r3=r3,
            r4=r4,
            cell_size=cell_size,
            eps=EPS,
        )

        broad_cell_candidate_count += broad_count
        capsule_candidate_count += capsule_count
        exact_test_count += candidate_indices.size

        max_broad_cell_candidates_one_row = max(
            max_broad_cell_candidates_one_row,
            broad_count,
        )

        max_capsule_candidates_one_row = max(
            max_capsule_candidates_one_row,
            capsule_count,
        )

        if candidate_indices.size == 0:
            continue

        # PIT Test

        hit_mask, details = swept_ctc_hits_points(
            points=points_now,
            p0=p0,
            p1=p1,
            u=u,
            h3=h3,
            h4=h4,
            r3=r3,
            r4=r4,
            h_0=h_0,
            candidate_indices=candidate_indices,
        )

        hit_indices = np.flatnonzero(hit_mask)

        for local_point_idx in hit_indices:
            point_ref_row_zero_based = int(point_ref_rows_now[local_point_idx])
            point_xyz = points_now[local_point_idx]

            hit_records.append({
                "Sweep ID": A_row_idx,
                "Waypoint": point_ref_row_zero_based,

                "Q1": q1,
                "Q2": q2,
                "Q3": q3,

                "Point X": point_xyz[0],
                "Point Y": point_xyz[1],
                "Point Z": point_xyz[2],

                "Cyl/Cone": details["section"][local_point_idx],
                "Axial Tip Dist.": details["axial"][local_point_idx],
                "Radial Dist.": details["radial"][local_point_idx],
                "Allowed Radius": details["allowed_radius"][local_point_idx],
            })

    loop_ms = (perf_counter_ns() - loop_start) / 1_000_000
    lap("Capsule spatial hash and exact CCD tests")

    hit_pairs = pd.DataFrame(hit_records)

    print("-" * 75)
    print(f"Rows in A:                         {A.shape[0]:,}")
    print(f"Rows in A_ref:                     {A_ref.shape[0]:,}")
    print(f"Unique A_ref points accumulated:   {n_test_points:,}")
    print(f"Spatial hash occupied cells:       {len(grid):,}")
    print(f"Broad cell candidates:             {broad_cell_candidate_count:,}")
    print(f"Capsule candidates:                {capsule_candidate_count:,}")
    print(f"Exact swept CTC tests:             {exact_test_count:,}")
    print(f"Positive swept CTC point hits:     {len(hit_pairs):,}")
    print(f"Max broad cell candidates / row:   {max_broad_cell_candidates_one_row:,}")
    print(f"Max capsule candidates / row:      {max_capsule_candidates_one_row:,}")
    print(f"Cell size:                         {cell_size:.6g}")
    print(f"Loop time:                         {loop_ms:.3f} ms")
    print(f"{'TOTAL':<45} {(perf_counter_ns() - t_start) / 1_000_000:10.3f} ms")

    if skipped_bad_q1 > 0:
        print(f"Skipped rows because q1 could not be used as an A_ref row index: {skipped_bad_q1:,}")

    return hit_pairs

In [28]:
hit_pairs = ccd_ctc_with_hash(folder_path=folder_path, h_0=h_0)

hit_pairs

A and A_ref Loaded from CSV                   /     0.003/ ms   total:      0.003 ms
Smallest capsule/sphere radius found: 21.6667
Median capsule/sphere radius used:    21.728
Largest capsule/sphere radius found:  22.3278
Capsule spatial hash cell size: 21.728
Capsule spatial hash and exact CCD tests      /  7606.109/ ms   total:   7606.112 ms
---------------------------------------------------------------------------
Rows in A:                         5,884
Rows in A_ref:                     5,000
Unique A_ref points accumulated:   4,992
Spatial hash occupied cells:       11
Broad cell candidates:             12,996,218
Capsule candidates:                17,685
Exact swept CTC tests:             17,685
Positive swept CTC point hits:     17
Max broad cell candidates / row:   4,973
Max capsule candidates / row:      400
Cell size:                         21.728
Loop time:                         7566.340 ms
TOTAL                                           7607.593 ms


,Sweep ID,Waypoint,Q1,Q2,Q3,Point X,Point Y,Point Z,Cyl/Cone,Axial Tip Dist.,Radial Dist.,Allowed Radius
0,5494,4532,4633.0,0.0,0.0,-5.82740,2.90654,4.180330,cone,0.489913,0.256168,0.507320
1,5494,4556,4633.0,0.0,0.0,-7.16029,3.02692,6.786230,cone,0.464371,0.115302,0.480870
2,5494,4557,4633.0,0.0,0.0,-6.23540,2.94237,5.855830,cone,0.478537,0.307898,0.495539
3,5494,4563,4633.0,0.0,0.0,-7.71950,3.06918,8.614120,cone,0.437063,0.189901,0.452592
4,5494,4564,4633.0,0.0,0.0,-8.66994,3.14444,9.569460,cone,0.410932,0.244011,0.425533
5,5494,4581,4633.0,0.0,0.0,-10.03890,3.22645,12.208600,cone,0.337110,0.307972,0.349088
6,5494,4582,4633.0,0.0,0.0,-9.05991,3.15862,11.226200,cone,0.373665,0.138752,0.386942
7,5494,4594,4633.0,0.0,0.0,-10.44070,3.23651,13.876800,cone,0.294485,0.069610,0.304948
8,5494,4606,4633.0,0.0,0.0,-11.56570,3.28464,16.269500,cone,0.212875,0.117186,0.220439
9,5495,3721,4633.0,1.0,0.0,-4.31353,2.75279,2.325790,cone,0.514152,0.472716,0.523205
